In [1]:
import re
import torch
from torch.utils.data import Dataset, DataLoader
import tiktoken

C:\Users\LENOVO\Downloads\Build-a-Large-Language-Model\.venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


# Tokenización y Embeddings en LLMs

En este notebook implementamos el pipeline básico de:
1. Tokenización del texto
2. Conversión a IDs
3. Creación de ventanas de contexto (sliding window)
4. Obtención de embeddings

Este proceso es fundamental en los LLMs porque los modelos no entienden texto directamente, sino vectores numéricos.

En sistemas agentic (agentes LLM que razonan y actúan), los embeddings permiten:
- Representar memoria
- Comparar significado entre textos
- Recuperar información relevante
- Razonar sobre contexto previo

Sin embeddings, no hay representación semántica computable.

In [3]:
import torch
import tiktoken

# Cargar texto
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# Tokenizador GPT-2
tokenizer = tiktoken.get_encoding("gpt2")

enc_text = tokenizer.encode(raw_text)
print("Total tokens:", len(enc_text))

Total tokens: 5145


## ¿Por qué tokenizamos?

Los modelos no procesan caracteres ni palabras directamente.
Procesan tokens.

Un token puede ser:
- Parte de una palabra
- Una palabra completa
- Signos de puntuación

Tokenizar permite:
- Reducir vocabulario
- Manejar palabras raras
- Reutilizar subestructuras

Esto hace el entrenamiento más eficiente y estable.

In [4]:
def create_dataset(tokens, max_length, stride):
    input_ids = []

    for i in range(0, len(tokens) - max_length, stride):
        input_chunk = tokens[i:i + max_length]
        input_ids.append(torch.tensor(input_chunk))

    return input_ids

max_length = 4
stride = 4

dataset = create_dataset(enc_text, max_length, stride)

print("Número de samples:", len(dataset))
print("Ejemplo sample:", dataset[0])

Número de samples: 1286
Ejemplo sample: tensor([  40,  367, 2885, 1464])


## Ventanas deslizantes y contexto

Los LLMs solo pueden procesar un número fijo de tokens (context window).

El uso de stride < max_length genera superposición (overlap).
Esto es importante porque:

- Permite que el modelo vea transiciones entre segmentos
- Mejora coherencia contextual
- Aumenta ejemplos de entrenamiento

Sin overlap, perderíamos relaciones entre frases consecutivas.

In [5]:
vocab_size = 50257
embedding_dim = 256

torch.manual_seed(123)

embedding_layer = torch.nn.Embedding(vocab_size, embedding_dim)

sample_input = dataset[0]
embedded = embedding_layer(sample_input)

print("Shape del embedding:", embedded.shape)

Shape del embedding: torch.Size([4, 256])


## ¿Por qué los embeddings codifican significado?

Un embedding es un vector denso aprendido por una red neuronal.

Durante entrenamiento:
- El modelo ajusta los vectores para minimizar error de predicción
- Palabras que aparecen en contextos similares terminan con vectores similares

Esto ocurre porque:
- Las capas lineales aprenden proyecciones
- El gradiente empuja vectores relacionados a regiones cercanas del espacio

Matemáticamente:
Un embedding es simplemente una matriz de pesos.
Seleccionar un token equivale a seleccionar una fila de esa matriz.

Relación con conceptos de redes neuronales:
- Es una capa lineal especial (lookup table)
- Se entrena con backpropagation
- Representa una proyección del espacio discreto (tokens) a uno continuo (R^d)

Por eso codifica significado:
Significado emerge como estructura geométrica en el espacio vectorial.
Distancia = similitud semántica.

In [6]:
# Experimento 1
max_length = 4
stride = 4
dataset1 = create_dataset(enc_text, max_length, stride)
print("Config 1 → samples:", len(dataset1))

# Experimento 2 (con overlap)
max_length = 4
stride = 2
dataset2 = create_dataset(enc_text, max_length, stride)
print("Config 2 → samples:", len(dataset2))

Config 1 → samples: 1286
Config 2 → samples: 2571


## Experimento: Cambio de stride

Con stride = max_length:
- No hay overlap
- Se generan menos muestras

Con stride < max_length:
- Hay superposición
- Se generan más muestras

¿Por qué?

Porque cada ventana comienza antes de que termine la anterior.
Esto aumenta los ejemplos de entrenamiento y mejora la captura de dependencias entre segmentos.

El overlap es útil porque:
- Mejora continuidad semántica
- Aumenta datos efectivos sin más texto
- Simula mejor el flujo natural del lenguaje